<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>


# 1. Inspect VSS Deployment

Part 1 runs on one pre-deployed Video Search and Summarization (VSS) stack with three perception services, each
with its own model. Other services store the video, carry the results and index them. Here you find out what
runs, check that it is ready, and see what the detector can and cannot report.

In this notebook you will:

- learn which service detects objects, which describes video and which searches it
- check that the services answer and their containers are running
- read the detector's labels from the running container
- see how the models share the two GPUs
- trace a video from upload to Elasticsearch

**Time:** 15 minutes. Run cells with **Shift+Enter**.


<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Part 1** &middot; about 15 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> Previous: [0. Course Onboarding](00-onboarding.ipynb) &middot; Next: [2. RT-CV: Detection & Tracking](02-rt-cv-detection-and-tracking.ipynb)


## 1.1 Learn what each service does

A traffic-monitoring application might need to count vehicles, describe a collision, and find other clips of
similar events. VSS provides a separate service, with its own model, for each of these jobs:

| Service | Model in this deployment | Output | What you can build with it |
|---|---|---|---|
| **RT-CV** (detection and tracking) | TrafficCamNet RT-DETR, a traffic detector, on GPU 0 | Object detections, tracklets and object embeddings | Counts, movement queries and searches by appearance |
| **RT-VLM** (captioning) | Cosmos 3 Super Reasoner (FP8), on GPU 0 | Text generated from a video window and a prompt | Scene descriptions and yes-or-no answers for alerts |
| **RT-Embed** (video search) | Cosmos-Embed1, 768-dimensional embeddings, on GPU 1 | One vector per video chunk | Search for moments that match a text description |

A detector reports what is in each frame. A question such as "did these two cars collide, or did one stop while
the other drove past?" is about how things change over a few seconds. Cosmos 3 Super Reasoner is a reasoning
vision language model (VLM), trained to answer that kind of question from a short clip. Its answer is still a
description, not proof, so you check it against the clip.

A fourth model runs on GPU 1: **Nemotron Nano 9B**, a text-only LLM that runs as an NVIDIA NIM (a packaged model
server). The VSS agent service (`vss-agent`) uses it to plan searches from questions asked in plain language. In
these notebooks you call the perception services directly, so the LLM stays idle. You still use the VSS agent
service: in notebook 2 it registers your uploads and starts the analysis.

At the end of Part 1 you use a different kind of agent: Codex, the coding agent you signed in to during
onboarding. You talk to it in a terminal, and it helps you build and test. The deployed VSS pipeline runs
without it.

VIOS stores the video, Kafka carries results between services, and Elasticsearch indexes them. An application can then get
back both the results and the original footage.


## 1.2 Check that the deployment is ready

Each service has an HTTP API. You call them with Python `requests`, starting with RT-CV's readiness endpoint.

From this notebook, every service sits behind the course proxy at `http://nginx`. The notebook runs in its own
container, so `localhost` would point back at the notebook itself.


In [ ]:
import requests

BASE = "http://nginx"   # the course proxy; localhost would be this notebook's own container

try:
    r = requests.get(BASE + "/rtvi-cv/api/v1/ready", timeout=10)
    print("RT-CV ready" if r.status_code == 200 else f"RT-CV not ready: HTTP {r.status_code}")
except requests.RequestException as exc:
    print(f"RT-CV not reachable yet ({type(exc).__name__}).")
print("Run the next cell to check the rest of the deployment.")


Now check the six main HTTP services and the eleven containers behind them. This is the check from section 0.3
of onboarding, as a single pass and without Kibana. The next cell prints `PASS` or `WAIT / CHECK` for each item.

You already waited for the stack there, so this pass should end with **READY TO CONTINUE**.


In [ ]:
import subprocess

SERVICES = {                                   # API routes the course calls, all through the proxy
    "RT-CV":         "/rtvi-cv/api/v1/ready",
    "RT-VLM":        "/rtvi-vlm/v1/health/ready",
    "RT-Embed":      "/rtvi-embed/v1/ready",
    "VIOS":          "/vios/vst/api/v1/sensor/list",
    "Elasticsearch": "/es/_cluster/health",
    "VSS agent":     "/agent/health",
}
REQUIRED = [                                   # containers on the data path (section 1.5)
    "vss-agent", "vss-rtvi-cv", "vss-rtvi-vlm", "vss-rtvi-embed",
    "vss-vios-ingress", "vss-vios-sensor", "vss-vios-streamprocessing",
    "elasticsearch", "kafka", "logstash", "vss-behavior-analytics",
]
INSPECT = "{{.Name}}|{{.State.Status}}|{{if .State.Health}}{{.State.Health.Status}}{{else}}no-healthcheck{{end}}"


def check_readiness():
    """One pass over the APIs and containers. Returns {name: (ok, detail)}."""
    results = {}
    for name, path in SERVICES.items():
        try:
            response = requests.get(BASE + path, timeout=10, allow_redirects=False)
            ok, detail = response.status_code == 200, f"HTTP {response.status_code}"
            if name == "Elasticsearch" and ok:
                health = response.json()
                ok = health.get("status") in ("green", "yellow") and not health.get("timed_out", False)
                detail += f", cluster {health.get('status')}"
        except (requests.RequestException, ValueError) as exc:
            ok, detail = False, type(exc).__name__
        results[name] = (ok, detail)
    try:
        result = subprocess.run(["docker", "inspect", "--format", INSPECT, *REQUIRED],
                                capture_output=True, text=True, timeout=30)
        states = {parts[0].lstrip("/"): parts[1:]
                  for line in result.stdout.splitlines() if len(parts := line.split("|")) == 3}
        for name in REQUIRED:
            state, health = states.get(name, ("missing", "unknown"))
            results[name] = (state == "running" and health in ("healthy", "no-healthcheck"), f"{state}, {health}")
        if result.returncode and not result.stdout.strip():
            results["docker"] = (False, f"docker inspect failed: {result.stderr.strip()[:120]}")
    except (OSError, subprocess.TimeoutExpired) as exc:
        results["docker"] = (False, f"Docker unavailable ({type(exc).__name__})")
    return results


results = check_readiness()
for name, (ok, detail) in results.items():
    print(f"{'PASS' if ok else 'WAIT / CHECK':12} {name:27} {detail}")
pending = [name for name, (ok, _) in results.items() if not ok]
lab_ready = not pending
if lab_ready:
    print("\nREADY TO CONTINUE: required APIs and pipeline containers passed.")
else:
    print("\nNOT READY: do not start video ingestion yet.")
    print("Needs attention:", ", ".join(pending))
    print("Wait a minute and run this cell again. If it stays NOT READY, follow the note below this cell.")


**READY TO CONTINUE** means the APIs respond and the pipeline containers are running with their health
checks passing. Containers without a health check are checked for running state only.

Elasticsearch can report `yellow` in this single-node lab, because its replica shards have no second node. That
passes. A `red` cluster does not. A 502 or a container marked `starting` means the stack is still coming up: wait
a minute and run the cell again. If it stays **NOT READY**, run the readiness check in section 0.3 of
[0. Course Onboarding](00-onboarding.ipynb) again. It waits up to 20 minutes and prints the deployer's status. If
that check stops with an error, tell the instructor.

Ready does not mean indexed. This check says the services answer. Notebook 2 checks that video actually flows
through them, with your own upload.


## 1.3 Identify the models and the detector's labels

Now check two rows of the table in section 1.1 against the running services: RT-VLM's model first, then RT-CV's
labels. RT-VLM lists its loaded model at `/rtvi-vlm/v1/models`. Read the id from the service, so your requests
always name the model that is actually loaded. The id the cell prints should contain `cosmos`.


In [ ]:
r = requests.get(BASE + "/rtvi-vlm/v1/models", timeout=20)
r.raise_for_status()
models = r.json().get("data") or []
if not models:
    raise RuntimeError(f"RT-VLM lists no models yet (the service is up but has not loaded its model): {r.text[:200]}")
vlm_model_id = models[0]["id"]
assert "cosmos" in vlm_model_id.lower(), f"unexpected RT-VLM model id {vlm_model_id!r}; this course expects Cosmos 3 Super Reasoner"
print("RT-VLM model id:", vlm_model_id)


**Predict before you run.** RT-CV runs TrafficCamNet RT-DETR, a traffic detector. Write down (a) how many
labels you expect in its label file, and (b) whether `truck`, `forklift`, `pedestrian` or `box` is among them.

The labels are in a file named `rtdetr-its-labels.txt`. The next cell reads the copy that the running detector
loaded, inside the RT-CV container (`vss-rtvi-cv`).


In [ ]:
# Read the label file that the running detector loaded, inside the RT-CV container
import subprocess

LABEL_FILE = "/opt/nvidia/deepstream/deepstream/sources/apps/sample_apps/metropolis_perception_app/configs/rtdetr-its-labels.txt"

proc = subprocess.run(["docker", "exec", "vss-rtvi-cv", "cat", LABEL_FILE],
                      capture_output=True, text=True, timeout=30)
if proc.returncode:
    raise RuntimeError(f"could not read {LABEL_FILE} in vss-rtvi-cv: "
                       f"{proc.stderr.strip() or 'no error text'}. Tell the instructor.")

labels = proc.stdout.split()
if "car" not in labels:
    raise RuntimeError(f"RT-CV is not running the traffic detector (labels: {labels}). Tell the instructor.")
print(len(labels), "labels:")
for label in labels:
    print("  ", label)


**Check your prediction.** Five labels: `background`, `bicycle`, `car`, `person` and `road_sign`. There is
no `truck`, `forklift` or `box`, and a pedestrian is a `person`. This list is the detector's whole vocabulary.
Every detection and every tracklet carries one of these five labels. An object outside the list is missed or
reported under one of them, however clearly it shows in the video.

<img src="../images/course/data-gap.png" width="960" alt="The detector's whole vocabulary, read from rtdetr-its-labels.txt, is five labels (background, bicycle, car, person, road_sign), and on a frame of traffic_short.mp4 both colliding vehicles get a hand-drawn car box. What we want to find, a collision or a stopped vehicle, has no label; the caption reads: Events the detector cannot name are found from tracklets or search, then described by the VLM.">

Events have no label at all. A collision or a stopped vehicle is something that happens, not an object, so RT-CV
cannot name it. In notebook 2 you find vehicles that stop by following their tracklets. In notebook 3 you ask
RT-VLM what happens in a clip. In notebook 4 you search for moments with RT-Embed. Even objects with a label can
be missed, so an empty result does not prove that nothing happened.


## 1.4 Inspect GPU usage

**Predict before you run.** The lab has two GPUs. From section 1.1: RT-CV and RT-VLM share GPU 0, and RT-Embed
and Nemotron Nano 9B share GPU 1. Which GPU do you expect to use more memory, and roughly how much of a 96 GB
card? Write your guess down, then run the cell.

`nvidia-smi` reports memory in MiB. Divide by 1,024 for GB.


In [ ]:
import subprocess

proc = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                      capture_output=True, text=True, timeout=30)
if proc.returncode:
    raise RuntimeError("nvidia-smi failed: " + proc.stderr.strip())
print(proc.stdout)


**Check your prediction.** GPU 0 uses more memory, and most of it belongs to one process. RT-VLM reserves
65 percent of the card when it starts: about 62 GB of a 96 GB H100 NVL. That covers the model weights, the KV
cache and working memory, whether or not anyone is asking for captions. The deployer sets this share with
`VSS_RT_VLM_GPU_MEMORY_UTILIZATION=0.65`. The detector next to it uses about 8 GB, so GPU 0 is
about three-quarters full. On GPU 1, Nemotron Nano 9B
holds about 38 GB and RT-Embed about 13 GB. Expect the exact numbers to move a little between launches.

<img src="../images/course/gpu-map.png" width="960" alt="Two memory bars, one per 96 GB H100 NVL: GPU 0 holds RT-VLM (Cosmos 3 Super Reasoner) with its weights, KV cache and working memory reserved up front at 0.65, about 62 GB, plus about 8 GB for RT-CV (TrafficCamNet RT-DETR); GPU 1 holds the idle Nemotron Nano 9B NIM, about 38 GB, and RT-Embed (Cosmos-Embed1), about 13 GB. The caption reads: Memory in use mostly reflects what servers reserved, not what they are doing.">

Memory in use shows what each server reserved, not how busy it is. It also does not tell you how many camera
streams the deployment can handle. That depends on the models, the video resolution, how many frames are sampled,
and the workload. RT-VLM takes the most memory of any service here. Notebook 3 ends with how to decide when to
call it, instead of running it on all of your video.


## 1.5 Trace the data path

Notebooks 2, 3 and 4 follow this map one block at a time: RT-CV, then RT-VLM, then RT-Embed. Read it now, so the
container list in the next section makes sense.

<img src="../images/course/data-path.gif" width="960" alt="Animated map of the data path: your notebook registers a video with vss-agent, VIOS stores it and feeds RT-CV (TrafficCamNet RT-DETR, GPU 0) and RT-Embed (Cosmos-Embed1, GPU 1), and RT-VLM (Cosmos 3 Super Reasoner, GPU 0) runs only when you ask, while the Nemotron Nano 9B NIM on GPU 1 stays idle. Results travel as Kafka topics through Behavior Analytics and Logstash into Elasticsearch, and your notebook reads the records from Elasticsearch and the clip from VIOS. The caption reads: Records from Elasticsearch, footage from VIOS: check one against the other.">

1. Your notebook registers a video through the VSS agent service (`vss-agent`).
2. VIOS (video storage) keeps the video and can cut any time window out of it as an MP4.
3. Registering a video starts RT-CV and RT-Embed automatically. RT-VLM runs only when you call it.
4. The services publish their results on Kafka (the message bus). Only metadata travels on Kafka. The video stays
   in VIOS.
5. Behavior Analytics turns RT-CV's tracklets into tracklet summaries: how each tracked object moved. It also filters
   RT-Embed's chunk embeddings for search. Logstash copies the results into Elasticsearch (the search index).
6. Your notebook reads records from Elasticsearch at `http://nginx/es` and the matching clip from VIOS.

To check a result, compare the record with its clip. An optional section at the end of notebook 2 adds a live
camera stream.


## 1.6 Inspect the running containers

The lab is connected to the Docker daemon that runs VSS, through `DOCKER_HOST`. So `docker` commands in this
notebook and in your terminal talk to the deployment, not to your own container. List the running containers.


In [ ]:
import subprocess

proc = subprocess.run(["docker", "ps", "--format", "{{.Names}}\t{{.Status}}"],
                      capture_output=True, text=True, timeout=60)
if proc.returncode:
    raise RuntimeError("docker ps failed: " + proc.stderr.strip())
print(proc.stdout)
print(len(proc.stdout.strip().splitlines()), "containers")


The number of containers depends on the configuration. Find each box of the map in the list:

| On the map | Container names |
|---|---|
| VSS agent service | `vss-agent` |
| VIOS | `vss-vios-ingress`, `vss-vios-sensor`, `vss-vios-streamprocessing` |
| RT-CV, RT-VLM, RT-Embed | `vss-rtvi-cv`, `vss-rtvi-vlm`, `vss-rtvi-embed` |
| Kafka, Logstash, Elasticsearch | `kafka`, `logstash`, `elasticsearch` |
| Behavior Analytics | `vss-behavior-analytics` |
| Nemotron Nano 9B | `nvidia-nemotron-nano-9b-v2` |

Use this list when you need a component's state or logs, for example `docker logs --tail 50 vss-rtvi-cv`. The
daemon runs the course's deployment. Read its state and logs, but never stop, restart, `down` or `prune` anything
on it. Never print a container's environment, and do not run a plain `docker inspect`, which prints it: those
environments hold the course key. The readiness check in section 1.2 asks `docker inspect` for each container's
state only.


## Check your understanding

1. You want to count vehicles passing a camera, and to check whether a stopped vehicle was in a collision. Which
   service do you start with for each job?
2. Can this detector identify a forklift as a forklift?
3. `nvidia-smi` shows GPU 0 about three-quarters full, but nobody has asked for a caption yet. Is RT-VLM busy?

<details>
<summary>Answers</summary>

1. RT-CV for the count: it gives detections and tracklets. Decide what counts as one vehicle passing, then check
   missed detections and identity changes against the video. RT-VLM for the collision: it describes a video
   window in answer to a prompt. Check its answer against the clip.
2. No. There is no forklift label, so the detector can only miss a forklift or report it under one of its five
   labels. To find events about objects without a label, search with RT-Embed and ask RT-VLM about the clips it
   finds.
3. No. RT-VLM reserves 65 percent of the card when it starts, for its weights, KV cache and working memory.
   Memory in use shows what a server reserved, not how busy it is or how many cameras the deployment can handle.

</details>


## Next

Open [2. RT-CV: Detection & Tracking](02-rt-cv-detection-and-tracking.ipynb), where you add a video to VIOS and
follow RT-CV's detections through Kafka into Elasticsearch. As you work, keep one use case of your own in mind:
which outputs would it need?


<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the cell in 1.3 that reads the detector's labels prints the label list.
>
> **Next: [2. RT-CV: Detection & Tracking](02-rt-cv-detection-and-tracking.ipynb)** &middot; Part 1 &middot; about 40 min  
> Add a video to VIOS, read RT-CV's detections from Kafka and Elasticsearch, and find the vehicles that stand still.
>
> [Back to Home](../00_Home.ipynb)


<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>
